In [0]:
# Storage access (move to Secret Scope for real deployments)
# Secrets are fetched at runtime — no hardcoded values.
import os

SECRET_SCOPE = "lakehouse-secrets"


def get_secret(key, default=None):
    """Fetch secret from Databricks Secret Scope, falling back to env var."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=key)
    except Exception:
        return os.getenv(key.upper().replace("-", "_"), default)


storage_account = os.getenv("AZURE_STORAGE_ACCOUNT", "your-storage-account")
storage_key = get_secret("storage-account-key", "")

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

# Path constants
SILVER_PATH = f"abfss://silver@{storage_account}.dfs.core.windows.net/crypto_trades"
GOLD_CHECKPOINT = f"abfss://gold@{storage_account}.dfs.core.windows.net/checkpoints/gold_crypto"
GOLD_TABLE = "hive_metastore.default.gold_crypto"

In [0]:
silver_df = (
    spark.readStream
         .format("delta")
         .load(SILVER_PATH)
)

In [0]:
from pyspark.sql.functions import col, avg, max, min, sum, count, window

# Watermark: bounds state so the engine discards tracking for windows
# older than 5 min past the latest event time. Without it, state grows forever.
#
# Tumbling windows: bucket trades into fixed 5-min intervals so downstream
# consumers get time-series metrics instead of a single ever-mutating row.
# Combined with append mode, each window is written once when finalized.

gold_df = (
    silver_df
    .withWatermark("trade_time", "5 minutes")
    .groupBy(
        window("trade_time", "5 minutes"),
        "symbol",
        "asset_class"
    )
    .agg(
        avg("trade_price").alias("avg_price"),
        max("trade_price").alias("max_price"),
        min("trade_price").alias("min_price"),
        sum("trade_quantity").alias("total_volume"),
        sum("trade_value").alias("total_trade_value"),
        count("*").alias("number_of_trades"),
        max("processing_time").alias("last_processed_time")
    )
)

In [0]:
# append mode: each finalized window is written once (no full-table rewrites).
# If migrating from complete mode, delete the old checkpoint directory first.
gold_query = (
    gold_df.writeStream
        .format("delta")
        .outputMode("append")
        .queryName("gold_crypto_aggregation")
        .option("checkpointLocation", GOLD_CHECKPOINT)
        .trigger(availableNow=True)
        .toTable(GOLD_TABLE)
)

gold_query.awaitTermination()

In [0]:
gold_table = spark.table(GOLD_TABLE)
print(f"✓ Gold record count: {gold_table.count():,}")

display(
    gold_table
        .select("window", "symbol", "avg_price", "total_volume", "number_of_trades")
        .orderBy(col("window.end").desc())
        .limit(10)
)